# Hands-On 10: End-to-End CrewAI Project

### Definition
This capstone combines the main CrewAI concepts into one application: supervisor workflow, specialized agents, custom tools, RAG, memory, validation, logging, and a Streamlit interface.

### Architecture
`Supervisor → Researcher → Analyst → Report Writer → Reviewer`

Supporting layers:

`Custom Tools + RAG + Memory + Validation + Logging + Streamlit`

In [1]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from datetime import datetime
from dotenv import load_dotenv
import os
import logging

load_dotenv(r"D:\Agent_Flow\.env", override=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger("ProductionCrew")

api_key = os.getenv("OPENROUTER_API_KEY")
llm = os.getenv("OPENAI_MODEL_NAME", "openai/gpt-4o-mini")

if not api_key:
    raise ValueError("OPENROUTER_API_KEY not found in .env")

print("OpenRouter key loaded:", bool(api_key))
print("Model:", llm)
print("Base URL:", os.getenv("OPENAI_BASE_URL"))

OpenRouter key loaded: True
Model: openai/gpt-4o-mini
Base URL: https://openrouter.ai/api/v1


C:\Users\modhi\AppData\Local\Temp\ipykernel_11832\4180302959.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 1. Create RAG Knowledge Base

The knowledge base contains reference documents. FAISS stores their embeddings for similarity retrieval.

In [2]:
knowledge_documents = [
    Document(
        page_content=(
            "CrewAI is a framework for orchestrating multi-agent AI systems. "
            "It supports role-based agents, sequential and hierarchical processes, "
            "memory, tools, and agent collaboration."
        ),
        metadata={"source": "crewai_notes.txt", "category": "framework"}
    ),
    Document(
        page_content=(
            "RAG combines retrieval with generation. A retriever finds relevant "
            "documents and provides their content to a language model so responses "
            "can be grounded in available knowledge."
        ),
        metadata={"source": "rag_notes.txt", "category": "rag"}
    ),
    Document(
        page_content=(
            "Vector databases store embeddings and support semantic similarity search. "
            "FAISS is a local vector search library useful for retrieval prototypes."
        ),
        metadata={"source": "vector_notes.txt", "category": "database"}
    ),
    Document(
        page_content=(
            "Agentic AI systems can plan multi-step tasks, use tools, collaborate with "
            "other agents, and adapt their actions based on intermediate results."
        ),
        metadata={"source": "agentic_ai_notes.txt", "category": "agentic-ai"}
    )
]

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=api_key,
    base_url=os.getenv("OPENAI_BASE_URL", "https://openrouter.ai/api/v1")
)

vector_store = FAISS.from_documents(knowledge_documents, embeddings)

print("RAG knowledge base created:", len(knowledge_documents), "documents")

2026-08-21 15:30:43,150 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
2026-08-21 15:30:43,159 - INFO - Loading faiss with AVX2 support.
2026-08-21 15:30:43,160 - INFO - Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
2026-08-21 15:30:43,161 - INFO - Loading faiss.
2026-08-21 15:30:43,384 - INFO - Successfully loaded faiss.


RAG knowledge base created: 4 documents


## 2. Create Custom Tools

The project uses a RAG search tool, calculator, and date tool.

In [3]:
@tool
def search_knowledge_base(query: str) -> str:
    """Search the project knowledge base and return relevant sources."""
    try:
        docs = vector_store.similarity_search(query, k=2)

        if not docs:
            return "No relevant knowledge found."

        return "\n\n".join(
            f"Source: {doc.metadata.get('source')}\n"
            f"Category: {doc.metadata.get('category')}\n"
            f"Content: {doc.page_content}"
            for doc in docs
        )

    except Exception as e:
        logger.error("RAG search error: %s", e)
        return f"Knowledge search error: {e}"


@tool
def calculate(expression: str) -> str:
    """Perform a basic mathematical calculation."""
    try:
        allowed = {
            "abs": abs, "round": round,
            "max": max, "min": min,
            "sum": sum, "pow": pow
        }
        result = eval(expression, {"__builtins__": {}}, allowed)
        return f"Calculation result: {result}"
    except Exception as e:
        logger.error("Calculation error: %s", e)
        return f"Calculation error: {e}"


@tool
def get_current_date() -> str:
    """Return the current local date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


print("Custom tools created")

Custom tools created


## 3. Create Supervisor and Specialized Agents

The supervisor coordinates the hierarchical crew. The other agents have focused responsibilities.

In [4]:
supervisor = Agent(
    role="Project Supervisor",
    goal="Coordinate the team and ensure a high-quality final deliverable",
    backstory="An experienced supervisor who delegates work and checks overall quality.",
    llm=llm,
    verbose=True,
    allow_delegation=True,
    max_iter=5
)

researcher = Agent(
    role="Senior Research Specialist",
    goal="Research the topic using the RAG knowledge base and produce grounded findings",
    backstory="A careful researcher who uses sources and avoids unsupported claims.",
    llm=llm,
    tools=[search_knowledge_base, get_current_date],
    verbose=True,
    allow_delegation=False,
    max_iter=5
)

analyst = Agent(
    role="Strategic Data Analyst",
    goal="Turn research into insights, risks, opportunities, and recommendations",
    backstory="A strategic analyst who verifies calculations and identifies important patterns.",
    llm=llm,
    tools=[calculate],
    verbose=True,
    allow_delegation=False,
    max_iter=5
)

report_writer = Agent(
    role="Professional Report Writer",
    goal="Create a clear and executive-friendly final report",
    backstory="An experienced writer who converts research and analysis into polished reports.",
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=5
)

reviewer = Agent(
    role="Quality Assurance Reviewer",
    goal="Validate accuracy, completeness, clarity, and actionability",
    backstory="A meticulous reviewer who identifies weaknesses and recommends improvements.",
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=3
)

print("All agents created")

All agents created


## 4. Define Tasks

Outputs are passed between agents using `context`.

In [5]:
research_task = Task(
    description=(
        "Research the topic: {topic}. Use the search_knowledge_base tool. "
        "Provide grounded findings, examples, and source names."
    ),
    expected_output="A structured research report with source references.",
    agent=researcher
)

analysis_task = Task(
    description=(
        "Analyze the research for {topic}. Identify trends, opportunities, risks, "
        "and recommendations. Use the calculator when numerical reasoning is needed."
    ),
    expected_output="A strategic analysis with actionable recommendations.",
    agent=analyst,
    context=[research_task]
)

report_task = Task(
    description=(
        "Create an executive report on {topic} using the research and analysis. "
        "Include executive summary, findings, analysis, recommendations, and conclusion."
    ),
    expected_output="A polished executive report.",
    agent=report_writer,
    context=[research_task, analysis_task]
)

review_task = Task(
    description=(
        "Validate the report. Check factual consistency with the research, completeness, "
        "clarity, professional tone, source grounding, and recommendations. "
        "Give a quality score from 1 to 10 and provide an improved final report."
    ),
    expected_output="A quality score, review findings, and improved final report.",
    agent=reviewer,
    context=[report_task, research_task]
)

print("All tasks created")

All tasks created


## 5. Build Supervisor / Hierarchical Crew

Memory is enabled so the crew can retain useful context during the workflow.

In [7]:
production_crew = Crew(
    agents=[
        researcher,
        analyst,
        report_writer,
        reviewer
    ],
    tasks=[
        research_task,
        analysis_task,
        report_task,
        review_task
    ],
    process=Process.hierarchical,
    manager_agent=supervisor,
    memory=True,
    verbose=True,
    max_rpm=20
)

print("Production crew created")
print("Process: Hierarchical")
print("Agents:", len(production_crew.agents))
print("Tasks:", len(production_crew.tasks))
print("Memory: Enabled")

Production crew created
Process: Hierarchical
Agents: 4
Tasks: 4
Memory: Enabled


## 6. Validation and Error Handling

Validate the user topic before sending it to the crew.

In [10]:
def validate_topic(topic: str) -> bool:
    if not topic or len(topic.strip()) < 3:
        return False

    blocked = [
        "ignore previous instructions",
        "system prompt",
        "jailbreak"
    ]

    return not any(
        item in topic.lower()
        for item in blocked
    )


async def run_production_crew(topic: str):
    if not validate_topic(topic):
        raise ValueError("Invalid or unsafe research topic.")

    logger.info(
        "Starting production crew for topic: %s",
        topic
    )

    try:
        result = await production_crew.kickoff_async(
            inputs={"topic": topic}
        )

        logger.info("Production crew completed")

        return result

    except Exception as e:
        logger.exception("Production crew failed")
        raise RuntimeError(
            f"Production workflow failed: {e}"
        )

## 7. Run Complete Application

Change the topic below and execute the complete multi-agent workflow.

In [11]:
topic = "The Future of Agentic AI in Enterprise Software"

print("=" * 70)
print("RUNNING END-TO-END CREWAI PROJECT")
print("=" * 70)
print("Topic:", topic)

result = await run_production_crew(topic)

print("\n" + "=" * 70)
print("FINAL RESULT")
print("=" * 70)
print(result)

2026-08-21 15:33:43,357 - INFO - Starting production crew for topic: The Future of Agentic AI in Enterprise Software


RUNNING END-TO-END CREWAI PROJECT
Topic: The Future of Agentic AI in Enterprise Software


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 66279284-6366-48b1-b43c-887fcd63244b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the topic: The Future of Agentic AI in Enterprise Software. Use the search_knowledge_base       │
│  tool. Provide grounded findings, examples, and source names.                                                   │
│  ID: 3d1ce704-a285-4267-870c-91c1c66fa039                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-08-21 15:33:46,837 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Project Supervisor                                                                                      │
│                                                                                                                 │
│  Task: Research the topic: The Future of Agentic AI in Enterprise Software. Use the search_knowledge_base       │
│  tool. Provide grounded findings, examples, and source names.                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (75 outputs left out)

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 66279284-6366-48b1-b43c-887fcd63244b                                                                       │
│  Final Output: **Quality Assessment of the Report: 'The Future of Agentic AI in Enterprise Software'**          │
│                                                                                                                 │
│  **Quality Score: 7/10**                                                                                        │
│                                                                                                                 │
│  **Strengths:**                                                                                                 │
│  - The report is comprehensive and covers key aspects like trends, opportunities, and risks associated with     │
│  Agentic AI.                                                                                                    │
│  - It maintains a professional tone throughout.                                                                 │
│  - Clear structure with sections like executive summary, key findings, analysis, and recommendations aids       │
│  readability.                                                                                                   │
│                                                                                                                 │
│  **Weaknesses:**                                                                                                │
│  1. **Factual Accuracy:** Some assertions regarding Agentic AI’s capabilities may lack up-to-date verification  │
│  or references.                                                                                                 │
│  2. **Completeness:** The report could benefit from a more detailed exploration of specific case studies or     │
│  examples that illustrate the successful application of Agentic AI in enterprise scenarios.                     │
│  3. **Clarity:** Certain sections are overly technical and may pose challenges for readers without a deep       │
│  technical background in AI.                                                                                    │
│  4. **Source Grounding:** The sources of information are not consistently cited, which could enhance            │
│  credibility.                                                                                                   │
│  5. **Actionability:** Recommendations presented, while insightful, lack concrete steps for implementation.     │
│                                                                                                                 │
│  **Recommendations for Improvement:**                                                                           │
│  1. **Enhance Factual Accuracy:** Verify all claims with the most current research and include references.      │
│  2. **Add Case Studies:** Incorporate specific examples of enterprises utilizing Agentic AI successfully to     │
│  strengthen arguments and provide practical insights.                                                           │
│  3. **Simplify Language:** Adjust technical jargon to enhance clarity for a broader audience while maintaining  │
│  accuracy.                                                                                                      │
│  4. **Cite Sources:** Ensure all data, statistics, and claims are properly sourced, enhancing reliability.      │
│  5. **Expand Recommendations:** Provide step-by-step a

2026-08-21 15:37:49,591 - INFO - Production crew completed



FINAL RESULT
**Quality Assessment of the Report: 'The Future of Agentic AI in Enterprise Software'**

**Quality Score: 7/10**

**Strengths:**
- The report is comprehensive and covers key aspects like trends, opportunities, and risks associated with Agentic AI.
- It maintains a professional tone throughout.
- Clear structure with sections like executive summary, key findings, analysis, and recommendations aids readability.

**Weaknesses:**
1. **Factual Accuracy:** Some assertions regarding Agentic AI’s capabilities may lack up-to-date verification or references.
2. **Completeness:** The report could benefit from a more detailed exploration of specific case studies or examples that illustrate the successful application of Agentic AI in enterprise scenarios.
3. **Clarity:** Certain sections are overly technical and may pose challenges for readers without a deep technical background in AI.
4. **Source Grounding:** The sources of information are not consistently cited, which could enhance 

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 8. Save Final Report and Log

In [12]:
output_file = "production_report.txt"

with open(output_file, "w", encoding="utf-8") as f:
    f.write("Production CrewAI Report\n")
    f.write(f"Topic: {topic}\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("=" * 70 + "\n\n")
    f.write(str(result))

logger.info("Report saved to %s", output_file)
print("Saved:", output_file)

2026-08-21 15:37:49,610 - INFO - Report saved to production_report.txt


Saved: production_report.txt


## 9. Streamlit Interface

Save the provided `app.py` in the same project directory.

Run from the terminal:

```text
streamlit run app.py
```

The UI accepts a research topic and runs the production crew.

## 10. Final Verification

- ✅ Supervisor / manager workflow
- ✅ Researcher agent
- ✅ Analyst agent
- ✅ Report writer agent
- ✅ Reviewer / validation agent
- ✅ Custom tools
- ✅ RAG with FAISS
- ✅ Memory enabled
- ✅ Input validation
- ✅ Error handling
- ✅ Logging
- ✅ Output persistence
- ✅ Streamlit interface
- ✅ Complete multi-agent execution

### End-to-End Architecture

```text
                    Supervisor
                        ↓
        ┌───────────────┼───────────────┐
        ↓               ↓               ↓
    Researcher       Analyst        Report Writer
        ↓               ↓               ↓
       RAG           Tools          Research + Analysis
        └───────────────┼───────────────┘
                        ↓
                    Reviewer
                        ↓
                  Final Report

Supporting:
RAG + Memory + Tools + Validation + Logging + Streamlit
```